<a href="https://colab.research.google.com/github/rmcN7/orion-t2/blob/main/fleet_filter_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/rmcN7/orion-t2/refs/heads/main/cleaned_opensky_data.csv?token=GHSAT0AAAAAAD76GOXBLX4KVODMMIICXMLQ2SMKBRQ"
df_tracks = pd.read_csv(url)
print(df_tracks.shape)
df_tracks.head()

(625, 17)


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,squawk,spi,position_source,pulled_at
0,151da3,NaN,Russian Federation,1783319092,1783319098,28.7910,41.3414,723.90,False,118.01,32.72,15.28,792.48,NaN,False,0,1783319101
1,151da3,AFL2139,Russian Federation,1783319399,1783319399,28.8860,41.6936,4274.82,False,179.17,40.46,0.00,4434.84,5157.0,False,0,1783319400
2,151da3,AFL2139,Russian Federation,1783319660,1783319700,29.4450,41.9157,7010.40,False,218.73,56.12,10.08,6880.86,5157.0,False,0,1783319704
3,151da3,AFL2139,Russian Federation,1783319796,1783319989,29.7531,42.0571,8915.40,False,238.75,95.44,6.83,8016.24,5157.0,False,0,1783320002
4,151da9,AFL2129,Russian Federation,1783316083,1783318494,38.0555,40.0866,11277.60,False,261.43,55.34,0.00,11163.30,737.0,False,0,1783318496


In [3]:
watch_countries = ["Russian Federation", "Belarus", "Syrian Arab Republic"]
flagged = df_tracks[df_tracks["origin_country"].isin(watch_countries)]
print(flagged.shape)

(20, 17)


In [4]:
flagged[["icao24", "callsign", "origin_country", "latitude", "longitude", "baro_altitude", "velocity"]]

,icao24,callsign,origin_country,latitude,longitude,baro_altitude,velocity
0,151da3,NaN,Russian Federation,41.3414,28.7910,723.90,118.01
1,151da3,AFL2139,Russian Federation,41.6936,28.8860,4274.82,179.17
2,151da3,AFL2139,Russian Federation,41.9157,29.4450,7010.40,218.73
3,151da3,AFL2139,Russian Federation,42.0571,29.7531,8915.40,238.75
4,151da9,AFL2129,Russian Federation,40.0866,38.0555,11277.60,261.43
5,151da9,AFL2129,Russian Federation,40.0866,38.0555,11285.22,261.43
6,151da9,AFL2129,Russian Federation,40.0866,38.0555,11285.22,261.43
7,151da9,AFL2129,Russian Federation,40.0866,38.0555,11277.60,261.43
8,152069,SVR773,Russian Federation,44.3617,33.6910,11582.40,4.74
9,152069,SVR773,Russian Federation,44.3617,33.6910,11582.40,4.74


In [5]:
def filter_by_country(df, countries):
    result = df[df["origin_country"].isin(countries)].copy()
    result["position_age_seconds"] = result["last_contact"] - result["time_position"]
    result["stale_position"] = result["position_age_seconds"] > 60
    return result

flagged = filter_by_country(df_tracks, watch_countries)
flagged[["icao24", "callsign", "origin_country", "latitude", "longitude",
         "velocity", "position_age_seconds", "stale_position"]]

,icao24,callsign,origin_country,latitude,longitude,velocity,position_age_seconds,stale_position
0,151da3,NaN,Russian Federation,41.3414,28.7910,118.01,6,False
1,151da3,AFL2139,Russian Federation,41.6936,28.8860,179.17,0,False
2,151da3,AFL2139,Russian Federation,41.9157,29.4450,218.73,40,False
3,151da3,AFL2139,Russian Federation,42.0571,29.7531,238.75,193,True
4,151da9,AFL2129,Russian Federation,40.0866,38.0555,261.43,2411,True
5,151da9,AFL2129,Russian Federation,40.0866,38.0555,261.43,2715,True
6,151da9,AFL2129,Russian Federation,40.0866,38.0555,261.43,3015,True
7,151da9,AFL2129,Russian Federation,40.0866,38.0555,261.43,3306,True
8,152069,SVR773,Russian Federation,44.3617,33.6910,4.74,113,True
9,152069,SVR773,Russian Federation,44.3617,33.6910,4.74,313,True


In [6]:
low_altitude = df_tracks[df_tracks["baro_altitude"] < 1000]
print(low_altitude.shape)
print(df_tracks["baro_altitude"].isna().sum(), "rows have missing altitude")

(75, 17)
16 rows have missing altitude


In [7]:
emergency_squawks = ["7500", "7600", "7700"]
emergency_flagged = df_tracks[df_tracks["squawk"].isin(emergency_squawks)]
print(emergency_flagged.shape)

(0, 17)


In [8]:
df_tracks["callsign_clean"] = df_tracks["callsign"].fillna("(no callsign)").str.strip()
df_tracks["callsign_clean"].head(10)

,callsign_clean
0,(no callsign)
1,AFL2139
2,AFL2139
3,AFL2139
4,AFL2129
5,AFL2129
6,AFL2129
7,AFL2129
8,SVR773
9,SVR773


In [14]:
def filter_by_country(df, countries):
    result = df[df["origin_country"].isin(countries)].copy()
    result["position_age_seconds"] = result["last_contact"] - result["time_position"]
    result["stale_position"] = result["position_age_seconds"] > 60
    result["callsign"] = result["callsign"].fillna("(no callsign)").str.strip()
    return result

In [16]:
flagged_full = filter_by_country(df_tracks, watch_countries)
flagged_full.to_csv("watchlist_flagged.csv", index=False)
print(flagged_full.shape)

(20, 19)


In [12]:
print(flagged_full["origin_country"].value_counts())
print(flagged_full.groupby("origin_country")[["latitude", "longitude", "baro_altitude"]].mean())
print(flagged_full["stale_position"].value_counts())

origin_country
Russian Federation      14
Belarus                  3
Syrian Arab Republic     3
Name: count, dtype: int64
                      latitude  longitude  baro_altitude
origin_country                                          
Belarus               40.38680  38.296100   10668.000000
Russian Federation    41.40255  35.488957    9420.497143
Syrian Arab Republic  40.36840  31.368633    8221.980000
stale_position
False    10
True     10
Name: count, dtype: int64


In [13]:
df_tracks = df_tracks.drop(columns=["callsign_clean"], errors="ignore")